# Article (Abstract + Fulltext) Screening Evaluation Against PERG Ground Truth

In [8]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

project_root = Path("../").resolve()
sys.path.insert(0, str(project_root))

import pandas as pd
from tqdm import tqdm

from utils.evals.create_perg_conditioned_files import add_perg_columns_from_screening

from sklearn.metrics import precision_recall_fscore_support

OUTPUT_DIR = Path('../data/agentslr/evals/article_screening')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

We evaluate screening as a binary article-level decision $y \in \{\text{INCLUDE}, \text{EXCLUDE}\}$ against the PERG reference label. Let $\text{TP}$ be articles correctly labelled INCLUDE, $\text{FP}$ be incorrect INCLUDE, and $\text{FN}$ be missed INCLUDE; then:

$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}, \quad
\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}, \quad
F_1 = \frac{2 \cdot P \cdot R}{P + R}
$$

where Precision measures the reliability of INCLUDE decisions and Recall measures how well we avoid prematurely assigning EXCLUDE to PERG-INCLUDE papers.

We report macro-$F_1$ to weight INCLUDE and EXCLUDE performance equally, rather than letting the majority class dominate.

## Abstract Screening

Abstract screening filters articles based on title and abstract content before full-text review.

In [2]:
def prep_screening_df(df, content_col, decision_col):
    df = df[df[content_col].notna()].reset_index(drop=True)
    df = df[df[decision_col] != "UNCLEAR"].reset_index(drop=True)
    return df

def compute_abstract_screening_metrics(pathogen, screened_path, perg_screened_path):
    df_screned = pd.read_csv(screened_path)
    df_perg = pd.read_csv(perg_screened_path)

    cols_to_be_added = ['perg_fulltext_result', 'perg_abstract_result', 'Covidence #', 'perg_subset']

    for col in cols_to_be_added:
        if col in df_screned.columns:
            df_screned = df_screned.drop(columns=col)
    
    df_abs = add_perg_columns_from_screening(df_screned, df_perg)

    df_abs = prep_screening_df(df_abs, "abstract", "ai4epi_abstract_decision")

    matched_rows = df_abs[df_abs.perg_subset==True]

    y_true = matched_rows['perg_abstract_result']
    y_pred = matched_rows['ai4epi_abstract_decision']

    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    return {
            "pathogen": pathogen,
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "n_eval": int(len(matched_rows)),
                }

def compute_all_abstract_screening(pathogens, agentslr_base, perg_base):
    all_metrics = []
    for pathogen in pathogens:
        screened_path = agentslr_base.format(pathogen=pathogen.lower())
        perg_screened_path = perg_base.format(pathogen=pathogen)
        metrics = compute_abstract_screening_metrics(pathogen, screened_path, perg_screened_path)
        all_metrics.append(metrics)

    if all_metrics:

        overall_metrics = {
            "pathogen": "Overall",
            "precision": float(pd.Series([m["precision"] for m in all_metrics]).mean()),
            "recall": float(pd.Series([m["recall"] for m in all_metrics]).mean()),
            "f1": float(pd.Series([m["f1"] for m in all_metrics]).mean()),
            "n_eval": int(pd.Series([m["n_eval"] for m in all_metrics]).sum()),
        }
        all_metrics.append(overall_metrics)

    return pd.DataFrame(all_metrics)

In [ ]:
models = ['oss', 'deepseek', 'kimi', 'glm', 'gpt']
pathogens = ['Marburg', 'Ebola', 'Lassa', 'SARS', 'Zika', 'MERS', 'Nipah']
df_all = []

for model_name in tqdm(models):
    agentslr_base = f'../data/agentslr/client/{model_name}/{{pathogen}}/screening/abstract_screening_results.csv'
    perg_base = f'../data/perg/screening/{{pathogen}}_filtered.csv'
    df_metrics = compute_all_abstract_screening(pathogens, agentslr_base, perg_base)
    df_metrics['model'] = model_name
    df_all.append(df_metrics)

df_abstract = pd.concat(df_all, ignore_index=True)
# df_abstract.to_csv(OUTPUT_DIR / 'abstract_screening_detailed.csv', index=False)

100%|██████████| 5/5 [02:32<00:00, 30.58s/it]


In [25]:
df_abstract_pivot = (
    df_abstract
    .pivot(index='pathogen', columns='model', values=['precision', 'recall', 'f1'])
    .swaplevel(0, 1, axis=1)
    .sort_index(axis=1)
    .reset_index()
)

overall_row = df_abstract_pivot[df_abstract_pivot['pathogen'] == 'Overall']
df_abstract_pivot = df_abstract_pivot[df_abstract_pivot['pathogen'] != 'Overall']
df_abstract_pivot = pd.concat([df_abstract_pivot, overall_row], ignore_index=True)
df_abstract_pivot

model pathogen  deepseek                           glm                      \
                      f1 precision    recall        f1 precision    recall   
0        Ebola  0.642130  0.802815  0.605102  0.774637  0.882649  0.721906   
1        Lassa  0.631555  0.844204  0.602009  0.728331  0.879944  0.679809   
2         MERS  0.651655  0.858080  0.602993  0.729305  0.887959  0.668215   
3      Marburg  0.578690  0.967965  0.552632  0.662733  0.878260  0.612022   
4        Nipah  0.527802  0.808603  0.538874  0.647052  0.897357  0.614888   
5         SARS  0.659349  0.820905  0.618902  0.781350  0.888399  0.728731   
6         Zika  0.655008  0.731963  0.628964  0.715063  0.760218  0.689605   
7      Overall  0.620884  0.833505  0.592782  0.719782  0.867827  0.673597   

model       gpt                          kimi                           oss  \
             f1 precision    recall        f1 precision    recall        f1   
0      0.680152  0.762190  0.644728  0.787964  0.786096  0.789868  0.746373   
1      0.664088  0.782547  0.630996  0.797470  0.838733  0.769148  0.753359   
2      0.669137  0.869276  0.616307  0.811810  0.859642  0.776527  0.777737   
3      0.620759  0.969735  0.578947  0.692833  0.785118  0.650512  0.688711   
4      0.594004  0.921897  0.578947  0.718604  0.853399  0.675946  0.696892   
5      0.653385  0.765352  0.618555  0.793304  0.803576  0.783986  0.772388   
6      0.642860  0.696159  0.622000  0.785239  0.778594  0.792555  0.746351   
7      0.646341  0.823880  0.612926  0.769603  0.815022  0.748363  0.740259   

model                      
      precision    recall  
0      0.743021  0.749869  
1      0.824801  0.716276  
2      0.832371  0.740285  
3      0.801150  0.643083  
4      0.842349  0.657149  
5      0.784312  0.761832  
6      0.731852  0.765953  
7      0.794265  0.719207

In [26]:
df_abstract_pivot.to_csv(OUTPUT_DIR / 'abstract_screening_summary.csv', index=False)

## Full-text Screening

For full-text screening, we use three evaluation configurations:

1. **AI abstract → AI full-text**: Any abstract EXCLUDE forces final EXCLUDE
2. **Human abstract → AI full-text** (PERG-conditioned): Any PERG abstract EXCLUDE forces final EXCLUDE
3. **AI direct full-text**: Evaluates the AI full-text decision without abstract gating

In [3]:
def compute_fulltext_screening_metrics(pathogen, screened_ft_path, screened_abs_path, perg_screened_path, mode='all'):
    df_ft = pd.read_csv(screened_ft_path)
    df_abs = pd.read_csv(screened_abs_path)
    df_perg = pd.read_csv(perg_screened_path)

    cols_to_drop = ['perg_fulltext_result', 'perg_abstract_result', 'Covidence #', 'perg_subset']
    for df in [df_ft, df_abs]:
        drop_cols = [c for c in cols_to_drop if c in df.columns]
        if drop_cols:
            df.drop(columns=drop_cols, inplace=True)

    df_ft = prep_screening_df(df_ft, "markdown_content", "ai4epi_fulltext_decision")
    df_abs = prep_screening_df(df_abs, "abstract", "ai4epi_abstract_decision")

    df_ft = add_perg_columns_from_screening(df_ft, df_perg)
    df_abs = add_perg_columns_from_screening(df_abs, df_perg)

    df_ft = df_ft[df_ft["perg_subset"] == True].reset_index(drop=True)

    if "ai4epi_abstract_decision" not in df_ft.columns:
        df_ft = df_ft.merge(
            df_abs[["article_id", "ai4epi_abstract_decision"]],
            on="article_id", how="left"
        )

    y_true = df_ft["perg_fulltext_result"]

    variants = {
        "fulltext_direct": df_ft["ai4epi_fulltext_decision"],
        "perg_conditioned": df_ft.apply(
            lambda r: "EXCLUDE" if r["perg_abstract_result"] == "EXCLUDE" else r["ai4epi_fulltext_decision"], axis=1
        ),
        "ai4epi_abstract_conditioned": df_ft.apply(
            lambda r: "EXCLUDE" if r["ai4epi_abstract_decision"] == "EXCLUDE" else r["ai4epi_fulltext_decision"], axis=1
        ),
    }

    selected = variants if mode == "all" else {mode: variants[mode]}

    result = {"pathogen": pathogen, "n_eval": len(df_ft)}
    for variant_name, y_pred in selected.items():
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
        result[f"precision_{variant_name}"] = float(precision)
        result[f"recall_{variant_name}"]    = float(recall)
        result[f"f1_{variant_name}"]        = float(f1)

    return result


def compute_all_fulltext_screening(pathogens, agentslr_ft_base, agentslr_abs_base, perg_base, mode='all'):
    all_metrics = []
    for pathogen in pathogens:
        metrics = compute_fulltext_screening_metrics(
            pathogen,
            screened_ft_path=agentslr_ft_base.format(pathogen=pathogen.lower()),
            screened_abs_path=agentslr_abs_base.format(pathogen=pathogen.lower()),
            perg_screened_path=perg_base.format(pathogen=pathogen),
            mode=mode,
        )
        all_metrics.append(metrics)

    if all_metrics:
        metric_cols = [k for k in all_metrics[0] if k not in ("pathogen", "n_eval")]
        overall = {"pathogen": "Overall", "n_eval": int(sum(m["n_eval"] for m in all_metrics))}
        for col in metric_cols:
            overall[col] = float(pd.Series([m[col] for m in all_metrics]).mean())
        all_metrics.append(overall)

    return pd.DataFrame(all_metrics)


In [5]:
pathogens = ['Marburg', 'Ebola', 'Lassa', 'SARS', 'Zika', 'MERS', 'Nipah']
models = ['oss', 'deepseek', 'kimi', 'glm', 'gpt']
df_all = []

for model_name in tqdm(models, desc="Processing models"):
    agentslr_ft_base = f'../data/agentslr/client/{model_name}/{{pathogen}}/screening/fulltext_screening_results.csv'
    agentslr_abs_base = f'../data/agentslr/client/{model_name}/{{pathogen}}/screening/abstract_screening_results.csv'
    perg_base = f'../data/perg/screening/{{pathogen}}_filtered.csv'

    df_metrics = compute_all_fulltext_screening(pathogens, agentslr_ft_base, agentslr_abs_base, perg_base, mode='all')
    df_metrics['model'] = model_name
    df_all.append(df_metrics)

df_full_text = pd.concat(df_all, ignore_index=True)

Processing models: 100%|██████████| 5/5 [03:51<00:00, 46.27s/it] 


In [6]:
df_full_text

,pathogen,n_eval,precision_fulltext_direct,recall_fulltext_direct,f1_fulltext_direct,precision_perg_conditioned,recall_perg_conditioned,f1_perg_conditioned,precision_ai4epi_abstract_conditioned,recall_ai4epi_abstract_conditioned,f1_ai4epi_abstract_conditioned,model
0,Marburg,799,0.643791,0.818027,0.694721,0.774577,0.828231,0.798877,0.745530,0.761565,0.753278,oss
1,Ebola,4129,0.667085,0.927787,0.720215,0.862132,0.971694,0.908508,0.731410,0.838585,0.772623,oss
2,Lassa,668,0.712800,0.911526,0.767490,0.832493,0.941413,0.877275,0.785954,0.780571,0.783231,oss
3,SARS,2047,0.635623,0.905030,0.676711,0.795954,0.953898,0.855257,0.708771,0.852149,0.757850,oss
4,Zika,2072,0.644766,0.849129,0.674077,0.805449,0.913662,0.848856,0.656442,0.788411,0.691742,oss
5,MERS,5729,0.692565,0.948489,0.764067,0.828065,0.960192,0.882303,0.757897,0.829885,0.789144,oss
6,Nipah,736,0.740464,0.881003,0.790817,0.891947,0.900483,0.896162,0.865661,0.841622,0.853166,oss
7,Overall,16180,0.676728,0.891570,0.726871,0.827231,0.924225,0.866748,0.750238,0.813255,0.771576,oss
8,Marburg,57,0.725490,0.609524,0.620584,0.725490,0.609524,0.620584,0.366071,0.488095,0.418367,deepseek
9,Ebola,508,0.796773,0.782134,0.785817,0.819910,0.797923,0.802757,0.681193,0.589497,0.553370,deepseek


In [9]:
df_full_text.rename(columns={
    'precision_ai4epi_abstract_conditioned': 'precision',
    'recall_ai4epi_abstract_conditioned': 'recall',
    'f1_ai4epi_abstract_conditioned': 'f1',
}, inplace=True)

df_full_text.to_csv(OUTPUT_DIR / 'fulltext_screening_detailed.csv', index=False)

In [10]:
df_full_text_pivot = (
    df_full_text
    .pivot(index='pathogen', columns='model', values=['precision', 'recall', 'f1'])
    .swaplevel(0, 1, axis=1)
    .sort_index(axis=1)
    .reset_index()
)

overall_row = df_full_text_pivot[df_full_text_pivot['pathogen'] == 'Overall']
df_full_text_pivot = df_full_text_pivot[df_full_text_pivot['pathogen'] != 'Overall']
df_full_text_pivot = pd.concat([df_full_text_pivot, overall_row], ignore_index=True)

df_full_text_pivot.to_csv(OUTPUT_DIR / 'fulltext_screening_summary.csv', index=False)

In [13]:
df_full_text

,pathogen,n_eval,precision_fulltext_direct,recall_fulltext_direct,f1_fulltext_direct,precision_perg_conditioned,recall_perg_conditioned,f1_perg_conditioned,precision,recall,f1,model
0,Marburg,799,0.643791,0.818027,0.694721,0.774577,0.828231,0.798877,0.745530,0.761565,0.753278,oss
1,Ebola,4129,0.667085,0.927787,0.720215,0.862132,0.971694,0.908508,0.731410,0.838585,0.772623,oss
2,Lassa,668,0.712800,0.911526,0.767490,0.832493,0.941413,0.877275,0.785954,0.780571,0.783231,oss
3,SARS,2047,0.635623,0.905030,0.676711,0.795954,0.953898,0.855257,0.708771,0.852149,0.757850,oss
4,Zika,2072,0.644766,0.849129,0.674077,0.805449,0.913662,0.848856,0.656442,0.788411,0.691742,oss
5,MERS,5729,0.692565,0.948489,0.764067,0.828065,0.960192,0.882303,0.757897,0.829885,0.789144,oss
6,Nipah,736,0.740464,0.881003,0.790817,0.891947,0.900483,0.896162,0.865661,0.841622,0.853166,oss
7,Overall,16180,0.676728,0.891570,0.726871,0.827231,0.924225,0.866748,0.750238,0.813255,0.771576,oss
8,Marburg,57,0.725490,0.609524,0.620584,0.725490,0.609524,0.620584,0.366071,0.488095,0.418367,deepseek
9,Ebola,508,0.796773,0.782134,0.785817,0.819910,0.797923,0.802757,0.681193,0.589497,0.553370,deepseek


In [16]:
# drop current 'precision' 'recall' 'f1' columns which are based on 'ai4epi_abstract_conditioned' variant
df_full_text_perg_conditioned = df_full_text.drop(columns=['precision', 'recall', 'f1'])

df_full_text_perg_conditioned = df_full_text_perg_conditioned.rename(columns={
    'precision_perg_conditioned': 'precision',
    'recall_perg_conditioned': 'recall',
    'f1_perg_conditioned': 'f1',
})


In [17]:


df_full_text_perg_conditioned_pivot = (
    df_full_text_perg_conditioned
    .pivot(index='pathogen', columns='model', values=['precision', 'recall', 'f1'])
    .swaplevel(0, 1, axis=1)
    .sort_index(axis=1)
    .reset_index()
)
overall_row = df_full_text_perg_conditioned_pivot[df_full_text_perg_conditioned_pivot['pathogen'] == 'Overall']
df_full_text_perg_conditioned_pivot = df_full_text_perg_conditioned_pivot[df_full_text_perg_conditioned_pivot['pathogen'] != 'Overall']
df_full_text_perg_conditioned_pivot = pd.concat([df_full_text_perg_conditioned_pivot, overall_row], ignore_index=True)
df_full_text_perg_conditioned_pivot.to_csv(OUTPUT_DIR / 'fulltext_screening_summary_perg_conditioned.csv', index=False)

## GPT-OSS-120B — Screening Ablation

This section computes ablation metrics for the OSS model across all three full-text screening configurations.

In [13]:
model_name = 'oss'
pathogens = ['Marburg', 'Ebola', 'Lassa', 'SARS', 'Zika', 'MERS', 'Nipah']

agentslr_ft_base  = f'../data/agentslr/client/{model_name}/{{pathogen}}/screening/fulltext_screening_results.csv'
agentslr_abs_base = f'../data/agentslr/client/{model_name}/{{pathogen}}/screening/abstract_screening_results.csv'
perg_base         = f'../data/perg/screening/{{pathogen}}_filtered.csv'

df_metrics_oss = compute_all_fulltext_screening(pathogens, agentslr_ft_base, agentslr_abs_base, perg_base, mode='all')

In [14]:
df_metrics_oss.to_csv(OUTPUT_DIR / 'gpt_oss_article_screening_ablations.csv', index=False)
df_metrics_oss

,pathogen,n_eval,precision_fulltext_direct,recall_fulltext_direct,f1_fulltext_direct,precision_perg_conditioned,recall_perg_conditioned,f1_perg_conditioned,precision_ai4epi_abstract_conditioned,recall_ai4epi_abstract_conditioned,f1_ai4epi_abstract_conditioned
0,Marburg,799,0.643791,0.818027,0.694721,0.774577,0.828231,0.798877,0.745530,0.761565,0.753278
1,Ebola,4129,0.667085,0.927787,0.720215,0.862132,0.971694,0.908508,0.731410,0.838585,0.772623
2,Lassa,668,0.712800,0.911526,0.767490,0.832493,0.941413,0.877275,0.785954,0.780571,0.783231
3,SARS,2047,0.635623,0.905030,0.676711,0.795954,0.953898,0.855257,0.708771,0.852149,0.757850
4,Zika,2072,0.644766,0.849129,0.674077,0.805449,0.913662,0.848856,0.656442,0.788411,0.691742
5,MERS,5729,0.692565,0.948489,0.764067,0.828065,0.960192,0.882303,0.757897,0.829885,0.789144
6,Nipah,736,0.740464,0.881003,0.790817,0.891947,0.900483,0.896162,0.865661,0.841622,0.853166
7,Overall,16180,0.676728,0.891570,0.726871,0.827231,0.924225,0.866748,0.750238,0.813255,0.771576


---

# Latex Tables (Beautify the computed metrics.)

In [1]:
import pandas as pd

def load_oss_screening_df(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df[df["model"] == "oss"][["pathogen", "precision", "recall", "f1"]].copy()
    df.columns = ["Pathogen", "Precision", "Recall", "F1-Score"]
    df = df.reset_index(drop=True)
    return df

CSV_PATH = "../data/agentslr/evals/article_screening/abstract_screening_detailed.csv"
df = load_oss_screening_df(CSV_PATH)
df

,Pathogen,Precision,Recall,F1-Score
0,Marburg,0.801150,0.643083,0.688711
1,Ebola,0.743021,0.749869,0.746373
2,Lassa,0.824801,0.716276,0.753359
3,SARS,0.784312,0.761832,0.772388
4,Zika,0.731852,0.765953,0.746351
5,MERS,0.832371,0.740285,0.777737
6,Nipah,0.842349,0.657149,0.696892
7,Overall,0.794265,0.719207,0.740259


In [2]:

def df_to_latex_screening_table(df: pd.DataFrame) -> str:
    body_rows = []
    overall_row = None

    for _, row in df.iterrows():
        line = f"{row['Pathogen']} & {row['Precision']:.2f} & {row['Recall']:.2f} & {row['F1-Score']:.2f} \\\\"
        if row["Pathogen"] == "Overall":
            overall_row = line
        else:
            body_rows.append(line)

    lines = [
        r"\renewcommand{\arraystretch}{1.1}",
        r"\setlength{\tabcolsep}{8pt}",
        r"\footnotesize",
        r"\begin{tabular}{l|ccc}",
        r"\toprule",
        r"\textbf{Pathogen} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\",
        r"\midrule",
        *body_rows,
        r"\midrule",
        overall_row,
        r"\bottomrule",
        r"\end{tabular}",
    ]

    return "\n".join(lines)

print(df_to_latex_screening_table(df))

\renewcommand{\arraystretch}{1.1}
\setlength{\tabcolsep}{8pt}
\footnotesize
\begin{tabular}{l|ccc}
\toprule
\textbf{Pathogen} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\midrule
Marburg & 0.80 & 0.64 & 0.69 \\
Ebola & 0.74 & 0.75 & 0.75 \\
Lassa & 0.82 & 0.72 & 0.75 \\
SARS & 0.78 & 0.76 & 0.77 \\
Zika & 0.73 & 0.77 & 0.75 \\
MERS & 0.83 & 0.74 & 0.78 \\
Nipah & 0.84 & 0.66 & 0.70 \\
\midrule
Overall & 0.79 & 0.72 & 0.74 \\
\bottomrule
\end{tabular}


### Model Ablations Table

In [5]:
import pandas as pd

def load_fulltext_screening_df(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    cols = {
        "pathogen": "Pathogen",
        "precision_ai4epi_abstract_conditioned": ("AI Abstract → AI Full-text", "Precision"),
        "recall_ai4epi_abstract_conditioned":    ("AI Abstract → AI Full-text", "Recall"),
        "f1_ai4epi_abstract_conditioned":        ("AI Abstract → AI Full-text", "F1"),
        "precision_perg_conditioned":            ("Human Abstract → AI Full-text", "Precision"),
        "recall_perg_conditioned":               ("Human Abstract → AI Full-text", "Recall"),
        "f1_perg_conditioned":                   ("Human Abstract → AI Full-text", "F1"),
        "precision_fulltext_direct":             ("AI Direct Full-text", "Precision"),
        "recall_fulltext_direct":                ("AI Direct Full-text", "Recall"),
        "f1_fulltext_direct":                    ("AI Direct Full-text", "F1"),
    }

    out = df[list(cols.keys())].copy()
    out.columns = pd.MultiIndex.from_tuples(
        [("", "Pathogen")] + [v for k, v in list(cols.items())[1:]]
    )
    return out.reset_index(drop=True)

CSV_PATH = "../data/agentslr/evals/article_screening/gpt_oss_article_screening_ablations.csv"
df = load_fulltext_screening_df(CSV_PATH)
df

AI Abstract → AI Full-text                      \
  Pathogen                  Precision    Recall        F1   
0  Marburg                   0.745530  0.761565  0.753278   
1    Ebola                   0.731410  0.838585  0.772623   
2    Lassa                   0.785954  0.780571  0.783231   
3     SARS                   0.708771  0.852149  0.757850   
4     Zika                   0.656442  0.788411  0.691742   
5     MERS                   0.757897  0.829885  0.789144   
6    Nipah                   0.865661  0.841622  0.853166   
7  Overall                   0.750238  0.813255  0.771576   

  Human Abstract → AI Full-text                     AI Direct Full-text  \
                      Precision    Recall        F1           Precision   
0                      0.774577  0.828231  0.798877            0.643791   
1                      0.862132  0.971694  0.908508            0.667085   
2                      0.832493  0.941413  0.877275            0.712800   
3                      0.795954  0.953898  0.855257            0.635623   
4                      0.805449  0.913662  0.848856            0.644766   
5                      0.828065  0.960192  0.882303            0.692565   
6                      0.891947  0.900483  0.896162            0.740464   
7                      0.827231  0.924225  0.866748            0.676728   

                       
     Recall        F1  
0  0.818027  0.694721  
1  0.927787  0.720215  
2  0.911526  0.767490  
3  0.905030  0.676711  
4  0.849129  0.674077  
5  0.948489  0.764067  
6  0.881003  0.790817  
7  0.891570  0.726871

In [6]:


def df_to_latex_fulltext_table(df: pd.DataFrame) -> str:
    def fmt(val):
        return f"{float(val):.2f}"

    body_rows = []
    overall_row = None

    for _, row in df.iterrows():
        pathogen = row[("", "Pathogen")]
        vals = [
            row[("AI Abstract → AI Full-text", "Precision")],
            row[("AI Abstract → AI Full-text", "Recall")],
            row[("AI Abstract → AI Full-text", "F1")],
            row[("Human Abstract → AI Full-text", "Precision")],
            row[("Human Abstract → AI Full-text", "Recall")],
            row[("Human Abstract → AI Full-text", "F1")],
            row[("AI Direct Full-text", "Precision")],
            row[("AI Direct Full-text", "Recall")],
            row[("AI Direct Full-text", "F1")],
        ]
        line = f"{pathogen} & " + " & ".join(fmt(v) for v in vals) + r" \\"
        if pathogen == "Overall":
            overall_row = line
        else:
            body_rows.append(line)

    lines = [
        r"\renewcommand{\arraystretch}{1.3}",
        r"\setlength{\tabcolsep}{9pt}",
        r"\footnotesize",
        r"\begin{tabular}{l|ccc|ccc|ccc}",
        r"\toprule",
        r"\multirow{2}{*}{\textbf{Pathogen}}",
        r"& \multicolumn{3}{c|}{\makecell{\textbf{AI Screen (Abstract)}\\$\rightarrow$\\\textbf{AI Screen (Full-text)}}}",
        r"& \multicolumn{3}{c|}{\makecell{\textbf{Human Screen (Abstract)}\\$\rightarrow$\\\textbf{AI Screen (Full-text)}}}",
        r"& \multicolumn{3}{c}{\makecell{~\\\textbf{AI Screen (Direct Full-text)}\\~}} \\",
        r"\cline{2-10}",
        r"& \textbf{Precision} & \textbf{Recall} & \textbf{F1}",
        r"& \textbf{Precision} & \textbf{Recall} & \textbf{F1}",
        r"& \textbf{Precision} & \textbf{Recall} & \textbf{F1} \\",
        r"\midrule",
        *body_rows,
        r"\midrule",
        overall_row,
        r"\bottomrule",
        r"\end{tabular}",
    ]

    return "\n".join(lines)


print(df_to_latex_fulltext_table(df))

\renewcommand{\arraystretch}{1.3}
\setlength{\tabcolsep}{9pt}
\footnotesize
\begin{tabular}{l|ccc|ccc|ccc}
\toprule
\multirow{2}{*}{\textbf{Pathogen}}
& \multicolumn{3}{c|}{\makecell{\textbf{AI Screen (Abstract)}\\$\rightarrow$\\\textbf{AI Screen (Full-text)}}}
& \multicolumn{3}{c|}{\makecell{\textbf{Human Screen (Abstract)}\\$\rightarrow$\\\textbf{AI Screen (Full-text)}}}
& \multicolumn{3}{c}{\makecell{~\\\textbf{AI Screen (Direct Full-text)}\\~}} \\
\cline{2-10}
& \textbf{Precision} & \textbf{Recall} & \textbf{F1}
& \textbf{Precision} & \textbf{Recall} & \textbf{F1}
& \textbf{Precision} & \textbf{Recall} & \textbf{F1} \\
\midrule
Marburg & 0.75 & 0.76 & 0.75 & 0.77 & 0.83 & 0.80 & 0.64 & 0.82 & 0.69 \\
Ebola & 0.73 & 0.84 & 0.77 & 0.86 & 0.97 & 0.91 & 0.67 & 0.93 & 0.72 \\
Lassa & 0.79 & 0.78 & 0.78 & 0.83 & 0.94 & 0.88 & 0.71 & 0.91 & 0.77 \\
SARS & 0.71 & 0.85 & 0.76 & 0.80 & 0.95 & 0.86 & 0.64 & 0.91 & 0.68 \\
Zika & 0.66 & 0.79 & 0.69 & 0.81 & 0.91 & 0.85 & 0.64 & 0.85 & 0.67 \\
